In [ ]:
! pip install lightgbm 

In [25]:
# imports

import pandas as pd
import numpy as np
import mlflow.sklearn
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTEENN
import optuna
from xgboost import XGBClassifier
import lightgbm as lgb

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [27]:
df = pd.read_csv("reddit_preprocessed.csv")
df.head()

,clean_comment,category,words,stop_words,characters,punctuation_chars
0,family mormon never tried explain still stare ...,1,39,13,259,0
1,buddhism much lot compatible christianity espe...,1,196,59,1268,0
2,seriously say thing first get complex explain ...,-1,86,40,459,0
3,learned want teach different focus goal not wr...,0,29,15,167,0
4,benefit may want read living buddha living chr...,1,112,45,690,0


In [26]:
import mlflow
mlflow.set_tracking_uri("http://ec2-18-117-120-209.us-east-2.compute.amazonaws.com:5000")


In [ ]:
mlflow.set_experiment("Training and Hyperparameter tuning LightGBM model")

In [22]:
# best parameters for vectorization and balancing found in previous notebooks
# -----------------
ngram_range = (1,2)
max_feature = 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_feature)

X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=0)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test) # correct way to do it, without data leakage

rus = RandomUnderSampler(random_state=0) 
X_train, y_train = rus.fit_resample(X_train, y_train)
# -----------------

# Function to log results in MLflow
def log_bestmodel_mlflow(model_name, model, X_train, X_test, y_train, y_test):

    with mlflow.start_run():

        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_undersampling_TFIDF_bigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")

def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = lgb.LGBMClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=0, verbosity=-1)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=30)

    best_params = study.best_params
    best_model = lgb.LGBMClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=0, verbosity=-1)

    log_bestmodel_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test)

    return best_params



In [23]:
best_params = run_optuna_experiment()

[I 2025-10-03 16:54:42,664] A new study created in memory with name: no-name-c90d985a-4c60-4191-8b4f-5b8a23e1e3da
c:\Users\iuri_\OneDrive\Desktop\youtube_sentiment_mlops_pipeline\myvenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-10-03 16:54:44,919] Trial 0 finished with value: 0.7429080444018089 and parameters: {'n_estimators': 134, 'learning_rate': 0.047549833285139335, 'max_depth': 7}. Best is trial 0 with value: 0.7429080444018089.
c:\Users\iuri_\OneDrive\Desktop\youtube_sentiment_mlops_pipeline\myvenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-10-03 16:54:46,091] Trial 1 finished with value: 0.5631081266273811 and parameters: {'n_estimators': 190, 'learning_rate': 0.0031944437432405728, 'max_depth': 3}. Best is trial 0 w

🏃 View run LightGBM_undersampling_TFIDF_bigrams at: http://ec2-18-117-120-209.us-east-2.compute.amazonaws.com:5000/#/experiments/381309362288343541/runs/e0302741bfd6457d968845ce4b9757d2
🧪 View experiment at: http://ec2-18-117-120-209.us-east-2.compute.amazonaws.com:5000/#/experiments/381309362288343541


In [29]:
best_params

{'n_estimators': 212, 'learning_rate': 0.09868989144653485, 'max_depth': 5}